### Test AI Agent 🤖 Tool Calling with DeepEval 🧪

Testing AI Agent involves testing of the Tools being invoked by an AI Agent. Here, AI Agent will invoke the necessary tools based on the given input and respond with the help of the tools being bounded with the AI Agent


<img src="./img/AIAGent.png" width="800" height="400" style="display: block; margin: auto;">

In [1]:
!uv pip install -qU duckduckgo-search

In [2]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    base_url="http://localhost:11434",
    model = "glm-4.7:cloud",
    temperature=0.5,
    max_tokens = 250
)

In [5]:
!uv pip install -U ddgs


Using Python 3.13.2 environment at: /Users/vaibhavarde/Desktop/TestAutomation/TestAutomationSkills/llmEvaluation/.venv
Resolved 4 packages in 1.44s                                         
Prepared 1 package in 0.57ms                                             
Uninstalled 1 package in 3ms
Installed 1 package in 4ms                                  
 - click==8.3.2
 + click==8.3.3


#### AI Agent with Tools

In [6]:
from langchain_core.tools import tool
from langchain_classic.agents import initialize_agent, AgentType
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

@tool
def add_numbers(a: int, b: int) -> int:
    "Add two numbers and return results."
    return int(a) + int(b)

@tool
def subtract_numbers(a: int, b: int) -> int:
    "Subtract two numbers and return results."
    return int(a) - int(b)

tools = [add_numbers, subtract_numbers, search_tool]

agent = initialize_agent(
    tools= tools,
    llm=llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    return_intermediate_steps=True
)

response = agent.invoke("Who is the current president of USA in 2025, just give the name")

print(response)




/var/folders/8h/zprf7hjs319_78816p34b90c0000gn/T/ipykernel_91898/1467121130.py:19: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the [LangGraph documentation](https://langchain-ai.github.io/langgraph/) as well as guides for [Migrating from AgentExecutor](https://python.langchain.com/docs/how_to/migrate_agent/) and LangGraph's [Pre-built ReAct agent](https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/).
  agent = initialize_agent(




> Entering new AgentExecutor chain...
```
{
  "action": "duckduckgo_search",
  "action_input": "current president of USA 2025"
}
```
Observation: The White House, official residence of the president of the United States The president of the United States is the head of state and head of government of the United States, [1] indirectly elected to a four-year term via the Electoral College. [2] Under the U.S. Constitution, the officeholder leads the executive branch of the federal government and is the commander-in-chief of the United ... The following is a list of events of the year 2025 in the United States. Following his election victory in November 2024, Donald Trump was inaugurated as the 47th President of the United States and began his second, nonconsecutive term on January 20. The 47th and current president of the United States is Donald John Trump. He was sworn into office on January 20, 2025. The United States has had 45 former U.S. presidents. Read about past presidents and v

In [7]:
def query_ai_agent(question):
    response = agent.invoke(question)
    intermediate_steps = response['intermediate_steps']
    agent_action, results = intermediate_steps[0]
    tool = agent_action.tool
    tool_input = agent_action.tool_input
    return response,tool, tool_input,

In [8]:
response,tool, tool_input = query_ai_agent("Who is the president of USA in 2025, just give me the name")

print(response)

print(tool)

print(tool_input)



> Entering new AgentExecutor chain...
```
{
  "action": "duckduckgo_search",
  "action_input": "president of USA 2025"
}
```
Observation: Bannon received 12% in the CPAC 2025 straw poll, coming second ahead of Ron DeSantis and Marco Rubio but far behind Vice President JD Vance (61%). Obama and Harris will further the decline of the USA as a world power as it becomes a godless, lawless nation. ... Presidential election , End Times ... In recent weeks, Donald Trump has disavowed authorship of Project 2025, a sweeping playbook to remake the federal government into a stronghold of ... ... boost to the global diplomacy of President Xi Jinping, the latest publication of trade data showing US$ 1.2 trillion trade surplus at the end of 2025 ... Ending the Gaza War (2025) • Within two weeks of taking office, President Trump proposed a peace plan to end the Gaza war, providing a fundamental ...
Thought:```
{
  "action": "Final Answer",
  "action_input": "Donald Trump"
}
```

> Finished chain.
{

### Testing AI Agent with DeepEval

In [9]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")

confidentAi = os.getenv("CONFIDENTAI")

In [10]:
import deepeval

deepeval.login(confidentAi)

🎉🥳 Congratulations! You've successfully logged in! 🙌

In [11]:
# !deepeval set-ollama deepseek-r1:8b
# !deepeval set-ollama gemma4:31b-cloud
!deepeval set-ollama --model gemma4:31b-cloud
# !deepeval set-ollama --model gpt-oss:120b-cloud 

🙌 Congratulations! You're now using a local Ollama model `gemma4:31b-cloud` for
all evals that require an LLM.


In [12]:
from deepeval.test_case import ToolCall

test_data = [
    {
        "input": "What is the sum of 20 and 40",
        "expected_output": "60",
        "tool_called": [
            ToolCall(name = "add_numbers")
        ]
    },
     {
        "input": "Who is the president of USA in 2025, just give me the name",
        "expected_output": "Donald Trump",
        "tool_called": [
            ToolCall(name = "duckduckgo_search")
        ]
    }
]

In [13]:
test_data

[{'input': 'What is the sum of 20 and 40',
  'expected_output': '60',
  'tool_called': [ToolCall(
       name="add_numbers"
   )]},
 {'input': 'Who is the president of USA in 2025, just give me the name',
  'expected_output': 'Donald Trump',
  'tool_called': [ToolCall(
       name="duckduckgo_search"
   )]}]

In [14]:
test_data[0]['tool_called']

[ToolCall(
     name="add_numbers"
 )]

In [16]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import ToolCorrectnessMetric

test_cases = []
for testcase in test_data:
  response, tool, tool_input = query_ai_agent(testcase['input'])
  test_case = LLMTestCase(
    input=testcase['input'],
    tools_called=[ToolCall(name=tool)],
    actual_output=response['output'],
    expected_tools=testcase['tool_called']
  )

  test_cases.append(test_case)





> Entering new AgentExecutor chain...
```
{
  "action": "add_numbers",
  "action_input": {
    "a": 20,
    "b": 40
  }
}
```
Observation: 60
Thought:```
{
  "action": "Final Answer",
  "action_input": "The sum of 20 and 40 is 60."
}
```

> Finished chain.


> Entering new AgentExecutor chain...
```
{
  "action": "duckduckgo_search",
  "action_input": "president of USA 2025"
}
```
Observation: 1 day ago - Donald Trump's second and current tenure as the president of the United States began upon his inauguration as the 47th president on January 20, 2025. Trump, a Republican, previously served as the 45th president from 2017 to 2021. He lost re-election to Democratic nominee Joe Biden in 2020, ... 1 week ago - The inauguration of Donald Trump as the 47th president of the United States took place on Monday, January 20, 2025. Due to freezing temperatures and high winds, it was held inside the U.S. Capitol rotunda in Washington, D.C. It was the 60th U.S. presidential inauguration and the s

In [17]:
test_cases

[LLMTestCase(input='What is the sum of 20 and 40', actual_output='The sum of 20 and 40 is 60.', expected_output=None, context=None, retrieval_context=None, additional_metadata=None, tools_called=[ToolCall(
     name="add_numbers"
 )], comments=None, expected_tools=[ToolCall(
     name="add_numbers"
 )], token_cost=None, completion_time=None, multimodal=False, name=None, tags=None, mcp_servers=None, mcp_tools_called=None, mcp_resources_called=None, mcp_prompts_called=None, custom_column_key_values=None),
 LLMTestCase(input='Who is the president of USA in 2025, just give me the name', actual_output='Donald Trump', expected_output=None, context=None, retrieval_context=None, additional_metadata=None, tools_called=[ToolCall(
     name="duckduckgo_search"
 )], comments=None, expected_tools=[ToolCall(
     name="duckduckgo_search"
 )], token_cost=None, completion_time=None, multimodal=False, name=None, tags=None, mcp_servers=None, mcp_tools_called=None, mcp_resources_called=None, mcp_prompts_

In [18]:
metrics = ToolCorrectnessMetric()

for testcase in test_cases:
    metrics.measure(test_case=testcase)
    print(metrics.score)
    print(metrics.reason)
    print(metrics.expected_tools)

/Users/vaibhavarde/Desktop/TestAutomation/TestAutomationSkills/llmEvaluation/.venv/lib/python3.13/site-packages/ric
h/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

1.0
[
	 Tool Calling Reason: All expected tools ['add_numbers'] were called (order not considered).
	 Tool Selection Reason: No available tools were provided to assess tool selection criteria
]

[ToolCall(
    name="add_numbers"
)]


1.0
[
	 Tool Calling Reason: All expected tools ['duckduckgo_search'] were called (order not considered).
	 Tool Selection Reason: No available tools were provided to assess tool selection criteria
]

[ToolCall(
    name="duckduckgo_search"
)]


[Confident AI Metric Data Log] Successfully posted metric data (0 metrics 
remaining in queue, 2 in flight) 
To disable dev logging, set CONFIDENT_METRIC_LOGGING_VERBOSE=0 as an environment
variable.
[Confident AI Metric Data Log] Successfully posted metric data (0 metrics 
remaining in queue, 1 in flight) 
To disable dev logging, set CONFIDENT_METRIC_LOGGING_VERBOSE=0 as an environment
variable.
